# VibeVoice Mystery Narration - Google Colab

This notebook demonstrates how to use Microsoft's VibeVoice 1.5B model for generating high-quality mystery podcast narration on Google Colab's free GPU.

## What is VibeVoice?

- State-of-the-art TTS for long-form conversational audio
- No token limits (64K context = ~50,000 words)
- Perfect for mystery storytelling (dramatic pauses, suspenseful tone)
- Can clone voices from 5-15 second samples
- MIT License (commercial use allowed)

## Requirements

- Google Colab with GPU runtime (free tier works!)
- 12GB+ GPU VRAM (T4, A100, or V100)
- ~10 minutes for setup + generation

---

## Step 1: Check GPU Availability

First, verify you have a GPU with sufficient VRAM:

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ GPU detected: {gpu_name}")
    print(f"✓ VRAM: {gpu_memory:.1f} GB")
    
    if gpu_memory < 12:
        print(f"\n⚠️  WARNING: GPU has only {gpu_memory:.1f}GB VRAM.")
        print("   Minimum 12GB recommended. Generation may fail.")
    else:
        print("\n✓ GPU has sufficient VRAM for VibeVoice!")
else:
    print("❌ ERROR: No GPU detected.")
    print("   Go to Runtime > Change runtime type > GPU")
    raise RuntimeError("GPU required for VibeVoice")

## Step 2: Install VibeVoice

Install the community fork of VibeVoice (official repo is disabled):

In [ ]:
%%bash
# Install VibeVoice community fork
pip install -q git+https://github.com/vibevoice-community/VibeVoice.git

# Install audio processing dependencies
pip install -q soundfile librosa

echo "✓ Installation complete!"

## Step 3: Prepare Your Mystery Story

Paste your mystery text here (can be very long - up to 50,000 words!):

In [ ]:
# Your mystery story text
MYSTERY_STORY = """
It was a fog-laden morning when I stumbled upon a peculiar bottle at a local 
estate sale in Philadelphia. Its emerald green hue caught the dim light, and 
embossed on its side were the words: SWAIM'S PANACEA PHILADA.

The bottle's antiquity was evident, but it was the unsettling aura surrounding 
it that piqued my curiosity. This was no ordinary tonic. It was the creation of 
William Swaim, a Philadelphia businessman who became one of the most notorious 
patent medicine purveyors of the 19th century.

The medicine's ingredients were potent—and dangerous. Its primary component, 
mercury dichloride, could cause serious illness with repeated use. Though some 
patients reported dramatic improvements, others suffered serious or even fatal 
consequences.

[PASTE YOUR FULL STORY HERE - can be thousands of words!]
""".strip()

# Show word count
word_count = len(MYSTERY_STORY.split())
print(f"Story length: {word_count} words")
print(f"Estimated audio: ~{word_count / 150:.1f} minutes")
print(f"Estimated generation time: ~{word_count / 150 * 6:.0f} minutes (0.15x real-time)")

## Step 4: Upload Voice Reference (Optional)

You can either:
1. **Upload a male narrator voice sample** (5-15 seconds, MP3/WAV)
2. **Skip this step** to use VibeVoice's default voice

To upload, click the folder icon on the left sidebar, then upload your audio file.

In [ ]:
import librosa
from pathlib import Path

# Option 1: Use uploaded voice reference
VOICE_REFERENCE_PATH = None  # Set to "/content/your_voice.mp3" if uploaded

# Option 2: Or use Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# VOICE_REFERENCE_PATH = "/content/drive/MyDrive/narrator_voice.mp3"

voice_audio = None
if VOICE_REFERENCE_PATH and Path(VOICE_REFERENCE_PATH).exists():
    voice_audio, sr = librosa.load(VOICE_REFERENCE_PATH, sr=24000, mono=True)
    duration = len(voice_audio) / sr
    print(f"✓ Loaded voice reference: {duration:.1f} seconds")
    
    if duration < 3:
        print("⚠️  WARNING: Reference audio <3 seconds. 5-15 seconds recommended.")
    elif duration > 30:
        print("⚠️  Trimming to first 15 seconds...")
        voice_audio = voice_audio[:int(15 * sr)]
else:
    print("ℹ️  No voice reference provided. Using VibeVoice default voice.")

## Step 5: Load VibeVoice Model

This downloads ~5.4GB model weights (takes 1-2 minutes):

In [ ]:
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
import torch

print("Loading VibeVoice processor...")
processor = VibeVoiceProcessor.from_pretrained("microsoft/VibeVoice-1.5B")
print("✓ Processor loaded")

print("\nLoading VibeVoice-1.5B model (5.4GB download)...")
print("This may take 1-2 minutes...")
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    "microsoft/VibeVoice-1.5B",
    torch_dtype=torch.bfloat16
).to("cuda")
print("✓ Model loaded on GPU")

# Set inference steps (5=fast, 20=quality)
INFERENCE_STEPS = 5  # Change to 10-20 for better quality
model.set_ddpm_inference_steps(num_steps=INFERENCE_STEPS)
print(f"✓ Inference steps: {INFERENCE_STEPS}")

print("\n✓ VibeVoice ready for generation!")

## Step 6: Generate Mystery Narration

This generates the full audio. **Be patient** - it takes ~6 minutes per minute of audio:

In [ ]:
import torch
from IPython.display import Audio, display
import soundfile as sf

# Configuration
CFG_SCALE = 1.3  # 1.0-2.0, higher = stricter text adherence

print("="*60)
print("GENERATING AUDIO")
print("="*60)

# Prepare inputs
voice_samples_formatted = [[voice_audio]] if voice_audio is not None else None

inputs = processor(
    text=[MYSTERY_STORY],
    voice_samples=voice_samples_formatted,
    return_tensors="pt"
)

# Move to GPU
inputs = {k: v.to("cuda") for k, v in inputs.items()}

# Estimate generation time
word_count = len(MYSTERY_STORY.split())
estimated_audio_minutes = word_count / 150
estimated_gen_time = estimated_audio_minutes * 6

print(f"\nEstimated audio length: ~{estimated_audio_minutes:.1f} minutes")
print(f"Estimated generation time: ~{estimated_gen_time:.0f} minutes")
print(f"CFG scale: {CFG_SCALE}")
print(f"\nGenerating... (this will take a while)")
print("☕ Go get coffee! ☕\n")

# Generate
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        cfg_scale=CFG_SCALE,
        tokenizer=processor.tokenizer,
        generation_config={'do_sample': False},
        verbose=True  # Show progress
    )

# Extract audio
audio_tensor = outputs.speech_outputs[0]
audio_array = audio_tensor.cpu().float().numpy()
sample_rate = 24000

actual_duration = len(audio_array) / sample_rate / 60

print("\n" + "="*60)
print("✓ GENERATION COMPLETE!")
print("="*60)
print(f"✓ Audio duration: {actual_duration:.1f} minutes")
print(f"✓ Sample rate: {sample_rate} Hz")

# Save audio
output_path = "/content/mystery_narration.wav"
sf.write(output_path, audio_array, sample_rate)
print(f"✓ Saved to: {output_path}")

# Play in notebook
print("\n🎧 Listen to your mystery narration:")
display(Audio(audio_array, rate=sample_rate))

## Step 7: Download Your Audio

Download the generated audio to your laptop:

In [ ]:
from google.colab import files

# Download the audio file
files.download('/content/mystery_narration.wav')

print("✓ Download started! Check your browser's download folder.")
print("\nNext steps:")
print("1. Listen to the audio and evaluate quality")
print("2. If satisfied, you can use this workflow for future stories")
print("3. Adjust CFG_SCALE (1.0-2.0) or INFERENCE_STEPS (5-20) for different quality")

## Tips for Best Results

### Voice Reference Quality
- **Length**: 5-15 seconds ideal
- **Quality**: Clean audio, no background noise
- **Content**: Natural speaking, not shouting/whispering
- **Male voice**: Works best with male narrator samples

### Generation Parameters
- **CFG Scale**:
  - 1.0 = More creative, natural variation
  - 1.3 = Balanced (recommended)
  - 2.0 = Stricter text adherence, less variation

- **Inference Steps**:
  - 5 steps = Fastest (2-3x faster than 20 steps)
  - 10 steps = Good balance
  - 20 steps = Best quality (slower)

### Text Preparation
- **Formatting**: Clean paragraphs, proper punctuation
- **Length**: Can handle up to ~50,000 words
- **Pacing**: Model automatically adds dramatic pauses
- **Emotion**: Suspenseful tone emerges naturally from content

---

## Cost

- **Colab Free Tier**: Completely free! (limited GPU hours per month)
- **Colab Pro ($10/month)**: More GPU hours, faster GPUs (A100)

**Typical usage**:
- 10-minute mystery: ~60 min GPU time
- 30-minute podcast: ~3 hours GPU time
- Free tier gives ~12 hours/week (enough for 2-3 episodes)

---

## Troubleshooting

**"CUDA out of memory"**
- Reduce INFERENCE_STEPS to 5
- Split very long stories into chapters
- Request different GPU type (Runtime > Change runtime)

**"Generation too slow"**
- Reduce INFERENCE_STEPS from 20 → 5 (2-3x faster)
- Use Colab Pro for A100 GPU (2x faster than T4)

**"Audio quality poor"**
- Increase INFERENCE_STEPS to 10-20
- Use higher quality voice reference
- Increase CFG_SCALE to 1.5-2.0

---

## Next Steps

After testing VibeVoice here:
1. If quality is excellent → Consider cloud GPU for production (RunPod/Vast.ai)
2. If quality is just OK → Stick with local XTTS/F5-TTS
3. Compare to your current engines (XTTS, Chatterbox, F5-TTS)

See `/docs/CLOUD_GPU_VIBEVOICE_SETUP.md` for production cloud GPU setup.

---

**Notebook created for**: Antique Mystery Podcast Generator  
**VibeVoice**: Microsoft Research (MIT License)  
**Community Fork**: https://github.com/vibevoice-community/VibeVoice